In [3]:
from tool1 import *
set_seed(42)
device = get_device()
print('device:', device)


device: cuda


In [5]:
from transformers import GPT2LMHeadModel, PreTrainedTokenizerFast

MODEL = 'skt/kogpt2-base-v2'
tok = PreTrainedTokenizerFast.from_pretrained(
    MODEL,
    bos_token='</s>',
    eos_token='</s>',
    unk_token='<unk>',
    pad_token='<pad>',
    mask_token='<mask>')
model = GPT2LMHeadModel.from_pretrained(MODEL).to(device)
model.eval()
print('vocab size:', tok.vocab_size)
print('params:', sum(p.numel() for p in model.parameters())/1e6, 'M')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer.json:   0%|          | 0.00/2.83M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/513M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/513M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab size: 51200
params: 164.485632 M


In [7]:
text = '안녕하세요. 저는 챗GPT입니다.'

ids = tok.encode(text)
print(text)
print('ids:', ids)
print('decoded:', tok.decode(ids))

안녕하세요. 저는 챗GPT입니다.
ids: [25906, 8702, 7801, 25856, 9265, 7162, 739, 8352, 413, 422, 426, 21154]
decoded: 안녕하세요. 저는 챗GPT입니다.


In [9]:
# Text Generation
prompt = '오늘의 날씨가 맑아서'
input_ids = tok.encode(prompt, return_tensors='pt').to(device)
with torch.no_grad():
    output = model.generate(input_ids, 
                            max_new_tokens=40,
                            do_sample=False,
                            pad_token_id=tok.pad_token_id)
print(tok.decode(output[0], skip_special_tokens=True))

오늘의 날씨가 맑아서 야외활동하기 좋겠습니다.
다만 아침에는 안개가 짙게 끼는 곳이 있겠습니다.
오늘 낮기온은 서울이 16도, 대전과 전주 17도, 대구 19도로 어제보다 조금 높겠습니다.



In [14]:
# Sampling
prompt = '매트릭스 세계관은'
input_ids = tok.encode(prompt, return_tensors='pt').to(device)
with torch.no_grad():
    for T in [0.7, 1.0, 1.5]:
        output = model.generate(
            input_ids,
            max_new_tokens=40,
            do_sample=True,
            temperature=T,
            top_k=50,
            top_p=0.9,
            pad_token_id=tok.pad_token_id
        )
        print(f'------ T={T} ------')
        print(tok.decode(output[0], skip_special_tokens=True))
        print()

------ T=0.7 ------
매트릭스 세계관은 기존 TV를 압도하는 수준이다.
실제 삼성전자는 지난해 TV에 이어 올해 TV에 이어 올해 TV에도 '퀀텀닷' 기술을 적용했다.
특히 올해는 미국 최대 가전

------ T=1.0 ------
매트릭스 세계관은 이미 중국 전역에 방영됐지만 전 세계 TV로 방영된 것은 이번이 처음이다.
CNN 등 외신에 따르면 CNN은 미 의회 소식통을 인용해 미 하원이 지난달 초

------ T=1.5 ------
매트릭스 세계관은 더 이상 세계관이나 다른 캐릭터처럼 볼 게 없다”며 “이들이 세계를 지배하게 되고 그 파괴의 결과는 전 세계로 확대됐다”고 말했다. 이 신문은 홍콩의 저명한 소식통들을



In [16]:
import numpy as np
prompt = '오늘은 날씨가 맑아서'
input_ids = tok.encode(prompt, return_tensors='pt').to(device)
with torch.no_grad():
    out = model(input_ids)
    logits = out.logits[0, -1]
    probs = F.softmax(logits, dim=-1)
    topk = torch.topk(logits, k=10)

for p, i in zip(topk.values, topk.indices):
    print(f'{tok.decode(i)}: {p.item():.4f} ({probs[i].item():.4f})')


야외: 12.9407 (0.0857)
나: 12.8814 (0.0808)
활동: 12.5270 (0.0567)
더: 12.1833 (0.0402)
바깥: 11.7753 (0.0267)
좋: 11.5571 (0.0215)
그런: 11.4450 (0.0192)
오늘: 11.1912 (0.0149)
낮: 10.9623 (0.0119)
추: 10.7750 (0.0098)
